<a href="https://colab.research.google.com/github/eltioxkypper/curso_ai_llama/blob/main/Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Asistente Inteligente Académico para Bases de Datos Distribuidas (RAG + Llama)

El Problema Real: Los estudiantes de nivel universitario en asignaturas avanzadas (como Bases de Datos Avanzadas) frecuentemente tienen dudas puntuales sobre fragmentación horizontal/vertical, replicación, concurrencia o arquitectura de sistemas distribuidos y móviles. Buscar esta información en manuales densos o presentaciones largas es tardado.

La Solución: Un asistente inteligente impulsado por Llama que combine:

RAG (Retrieval-Augmented Generation): Para consultar directamente el material didáctico oficial (como los PDFs y presentaciones de clase sobre fragmentación y transacciones móviles).

Fine-Tuning / Estilo Pedagógico: Configurado con un tono formal, académico, directo y estructurado (alineado al perfil de instructor universitario).

In [5]:
!pip install -q langchain langchain-community langchain-text-splitters chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Texto de ejemplo basado en el material didáctico del curso de Bases de Datos Avanzadas
material_estudio = [
    """La fragmentación horizontal consiste en dividir una relación r en n fragmentos r1, r2, ..., rn mediante una condición lógica (como selección), donde cada tupla se asigna a un fragmento específico. Ventajas: permite procesamiento en paralelo y acerca los datos a donde más se consultan.""",
    """La fragmentación vertical divide el esquema de una relación r en esquemas más pequeños. Todos los esquemas deben contener una llave o superllave candidata para asegurar la propiedad de join sin pérdida de información.""",
    """La replicación de datos implica almacenar copias redundantes de una relación o fragmento en diferentes sitios para mejorar la disponibilidad y permitir tolerancia a fallas ante la caída de un nodo.""",
    """Las transacciones móviles se caracterizan por tolerar desconexiones frecuentes, recursos limitados de hardware, bajo ancho de banda y la necesidad de sincronización y control de concurrencia optimista o pesimista al reconectarse."""
]

# Procesamiento y vectorización
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.create_documents(material_estudio)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("¡Base de conocimiento RAG indexada correctamente!")

/tmp/ipykernel_3182/363008713.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_3182/363008713.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

¡Base de conocimiento RAG indexada correctamente!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Asegurar que el modelo y tokenizador estén cargados en memoria
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Cargando modelo y tokenizador (esto tardará solo unos segundos)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("¡Modelo listo!")

# 2. Re-verificar la base de conocimiento RAG por seguridad
material_estudio = [
    """La fragmentación horizontal consiste en dividir una relación r en n fragmentos r1, r2, ..., rn mediante una condición lógica (como selección), donde cada tupla se asigna a un fragmento específico. Ventajas: permite procesamiento en paralelo y acerca los datos a donde más se consultan.""",
    """La fragmentación vertical divide el esquema de una relación r en esquemas más pequeños. Todos los esquemas deben contener una llave o superllave candidata para asegurar la propiedad de join sin pérdida de información.""",
    """La replicación de datos implica almacenar copias redundantes de una relación o fragmento en diferentes sitios para mejorar la disponibilidad y permitir tolerancia a fallas ante la caída de un nodo.""",
    """Las transacciones móviles se caracterizan por tolerar desconexiones frecuentes, recursos limitados de hardware, bajo ancho de banda y la necesidad de sincronización y control de concurrencia optimista o pesimista al reconectarse."""
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.create_documents(material_estudio)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Función del asistente optimizada y rápida
def asistente_academico(pregunta: str):
    docs_recuperados = retriever.invoke(pregunta)
    contexto = "\n".join([doc.page_content for doc in docs_recuperados])

    prompt = f"""[INST] Actúa como un asistente académico experto en Bases de Datos Avanzadas.
Responde a la pregunta utilizando exclusivamente el contexto proporcionado. Mantén un tono formal, claro y estructurado.

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta: [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    respuesta_completa = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return respuesta_completa.split("[/INST]")[-1].strip()

# 4. Prueba de ejecución
consulta = "¿Cuáles son las ventajas de la fragmentación horizontal?"
print(f"\nPregunta: {consulta}\n")
print("Generando respuesta...")
print(f"\nRespuesta del Asistente:\n{asistente_academico(consulta)}")

Cargando modelo y tokenizador (esto tardará solo unos segundos)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

¡Modelo listo!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Pregunta: ¿Cuáles son las ventajas de la fragmentación horizontal?

Generando respuesta...
